In [1]:
from datasets import Dataset
import json

train_file_path = "../new_data/train_large.json"
val_file_path = "../new_data/val_large.json"
test_file_path = "../new_data/test_large.json"

# Chuyển dữ liệu thành định dạng phù hợp cho Hugging Face Dataset
data_processed = []
# Đọc file JSON
with open(train_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_processed.append({
            "input": data["text"],
            "output": data["label"]
        })

data_validate = []
# Đọc file JSON
with open(val_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_validate.append({
            "input": data["text"],
            "output": data["label"]
        })

data_test = []
# Đọc file JSON
with open(test_file_path, 'r', encoding='utf-8') as file:
    for line in file:
        data = json.loads(line.strip())
        data_test.append({
            "input": data["text"],
            "output": data["label"]
        })
# Chuyển đổi dữ liệu thành Dataset của Hugging Face
train_data = Dataset.from_dict({
    'input': [item['input'] for item in data_processed],
    'output': [item['output'] for item in data_processed]
})

val_data = Dataset.from_dict({
    'input': [item['input'] for item in data_validate],
    'output': [item['output'] for item in data_validate]
})

test_data = Dataset.from_dict({
    'input': [item['input'] for item in data_test],
    'output': [item['output'] for item in data_test]
})


/home/creator/miniconda3/envs/toanpn/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

# Hiển thị 10 phần tử đầu tiên
print("10 phần tử đầu tiên trong dataset:")
for i in range(20):
    print(f"{train_data[i]}")

10 phần tử đầu tiên trong dataset:
{'input': 'MS. MYERS I believe once during the day and once last night.', 'output': 'english'}
{'input': 'I kind of doubt that any revolution, armed or otherwise, was ever started without vast amounts of failed working within a system.', 'output': 'english'}
{'input': 'Now, I know full well what coins areused every day in Canada.', 'output': 'english'}
{'input': 'I think this guy is going to be just a little bit disappointed.', 'output': 'english'}
{'input': 'Be calm, respectful, and friendly secretarieshave more power than you might realize, and you never knowcould be the dean of admissions answering the phone.', 'output': 'english'}
{'input': 'Unfortunately, the intervention was too late at least for some of the victims.', 'output': 'english'}
{'input': 'How should we execute you', 'output': 'english'}
{'input': 'Guessyou can choose either buggy stateoftheart stuff, or robust averagestuff For my particular configuration tower, 300 watt supply, pkg3,

In [3]:
from torch.utils.data import Dataset as dt

class LanguageDataset(dt):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_text = self.data[idx]["input"]
        target_text = self.data[idx]["output"]

        # Mã hóa input và target
        input_ids = self.tokenizer(
            input_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )
        target_ids = self.tokenizer(
            target_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

        return {
            "input_ids": input_ids["input_ids"].squeeze(0),
            "attention_mask": input_ids["attention_mask"].squeeze(0),
            "labels": target_ids["input_ids"].squeeze(0)
        }


In [4]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import DataLoader

# Load pre-trained model and tokenizer
model_name = 't5-small'  # Or use 't5-base' or 't5-large' for more capacity
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

/home/creator/.local/lib/python3.10/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/home/creator/.local/lib/python3.10/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in

In [5]:
# Áp dụng hàm tiền xử lý cho dữ liệu
train_dataset = LanguageDataset(train_data, tokenizer)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = LanguageDataset(val_data, tokenizer)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True)

test_dataset = LanguageDataset(test_data, tokenizer)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True)

In [6]:
print(train_dataset[0])
print(f"{len(train_dataset)}")

{'input_ids': tensor([5266,    5,  283,  476, 9984,   27,  857,  728,  383,    8,  239,   11,
         728,  336,  706,    5,    1,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0

In [7]:
from transformers import AdamW, get_linear_schedule_with_warmup, T5ForConditionalGeneration

import torch
from tqdm import tqdm  # Dùng để hiển thị thanh tiến độ trong huấn luyện


# Load pre-trained model and tokenizer
model_name = 't5-small'  # Or use 't5-base' or 't5-large' for more capacity
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Tạo optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# số epoch
epochs = 3

# Cấu hình học tập
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Định nghĩa số bước warmup và số bước cập nhật learning rate
num_training_steps = len(train_dataloader) * epochs
num_warmup_steps = int(0.1 * num_training_steps)

# Scheduler điều chỉnh learning rate
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch + 1} - Average Loss: {avg_loss:.4f}")


/home/creator/miniconda3/envs/toanpn/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
Epoch 1/3: 100%|██████████| 4254/4254 [1:12:19<00:00,  1.02s/it]


Epoch 1 - Average Loss: 0.8035


Epoch 2/3: 100%|██████████| 4254/4254 [1:12:15<00:00,  1.02s/it]


Epoch 2 - Average Loss: 0.0025


Epoch 3/3: 100%|██████████| 4254/4254 [1:12:09<00:00,  1.02s/it]

Epoch 3 - Average Loss: 0.0017


In [11]:
# Lưu mô hình dưới dạng SavedModel
model.save_pretrained('../my_t5_model')
tokenizer.save_pretrained('../my_t5_model')  # Lưu luôn tokenizer để tái sử dụng

('../my_t5_model/tokenizer_config.json',
 '../my_t5_model/special_tokens_map.json',
 '../my_t5_model/spiece.model',
 '../my_t5_model/added_tokens.json')

In [30]:
import torch
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup

# Khởi tạo danh sách để lưu nhãn thực tế và dự đoán
y_true = []
y_pred = []
loss_values = []

# Chuyển model sang chế độ đánh giá
model.eval()

# Thiết bị tính toán
device = torch.device("cuda")
model.to(device)

# Tính Loss và dự đoán
# Không tính toán gradient trong khi đánh giá
with torch.no_grad():
    for batch in tqdm(val_dataloader, desc="Validating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        # Tính Loss
        loss = outputs.loss
        loss_values.append(loss.item())  # Lưu giá trị loss

        # Generate dự đoán
        pred_ids = model.generate(input_ids)
        for i in range(len(pred_ids)):
            pred_text = tokenizer.decode(pred_ids[i], skip_special_tokens=True)
            labels_decode = tokenizer.decode(labels[i], skip_special_tokens=True)
            # Lưu nhãn thực tế và dự đoán
            y_true.append(labels_decode)
            y_pred.append(pred_text)


Validating: 100%|██████████| 1216/1216 [21:52<00:00,  1.08s/it]


In [31]:
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_true_encoded = label_encoder.fit_transform(y_true)
y_pred_encoded = label_encoder.transform(y_pred)

In [32]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Accuracy
accuracy = accuracy_score(y_true_encoded, y_pred_encoded)

# Macro-F1
macro_f1 = f1_score(y_true_encoded, y_pred_encoded, average='macro')

# Weighted-F1
weighted_f1 = f1_score(y_true_encoded, y_pred_encoded, average='weighted')

# Confusion Matrix
cm = confusion_matrix(y_true_encoded, y_pred_encoded)

# Tính Loss trung bình
average_loss = sum(loss_values) / len(loss_values)

# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)
print("Average Loss:", average_loss)


Accuracy: 0.9305698415963793
Macro-F1: 0.9074067061353355
Weighted-F1: 0.9322596852632946
Confusion Matrix:
 [[37458    55     0]
 [  108 21673  5099]
 [   47    91 13245]]
Average Loss: 0.0013651721965169073


In [37]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder

def predict_language(input_text, model, tokenizer):
    model.eval()
    input_ids = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )["input_ids"].to(device)

    # Sinh đầu ra
    outputs = model.generate(input_ids)
    predicted_language = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return predicted_language

test_y_true = []
test_y_pred = []

for data in test_data:
    predicted = predict_language(data['input'], model, tokenizer)
    # Lưu nhãn thực tế và dự đoán
    test_y_true.append(data['output'])
    test_y_pred.append(predicted)


# Encode labels
label_encoder = LabelEncoder()
test_y_true_encoded = label_encoder.fit_transform(test_y_true)
test_y_pred_encoded = label_encoder.transform(test_y_pred)
# Accuracy
accuracy = accuracy_score(test_y_true_encoded, test_y_pred_encoded)

# Macro-F1
macro_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='macro')

# Weighted-F1
weighted_f1 = f1_score(test_y_true_encoded, test_y_pred_encoded, average='weighted')

# Confusion Matrix
cm = confusion_matrix(test_y_true_encoded, test_y_pred_encoded)

# In kết quả
print("Accuracy:", accuracy)
print("Macro-F1:", macro_f1)
print("Weighted-F1:", weighted_f1)
print("Confusion Matrix:\n", cm)

Accuracy: 0.9279487772892078
Macro-F1: 0.9047458396629905
Weighted-F1: 0.9296913780878849
Confusion Matrix:
 [[18604    27     0]
 [   53 10790  2646]
 [   30    46  6693]]


In [ ]:
import torch

# Lưu ví dụ cho từng case
cases = {
    "Case 1": None,  # Thực tế 0, dự đoán 0
    "Case 2": None,  # Thực tế 0, dự đoán 1
    "Case 3": None,  # Thực tế 1, dự đoán 1
    "Case 4": None,  # Thực tế 1, dự đoán 2
    "Case 5": None,  # Thực tế 2, dự đoán 2
    "Case 6": None   # Thực tế 2, dự đoán 1
}
texts = []
test_labels = []
pred_labels = []
# Loop qua dữ liệu test
for data in tqdm(test_data, desc="Testing"):
    texts.append(data['input'])
    predicted = predict_language(data['input'], model, tokenizer, device)
    test_labels.append(test_label_map[data['output']])
    pred_labels.append(predicted)


In [ ]:
# Lấy ví dụ cho từng case
for idx, (text, true_label, pred_label) in enumerate(zip(test_texts, test_labels, predictions)):
    if true_label == 0 and pred_label == 0 and cases["Case 1"] is None:
        cases["Case 1"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 0 and pred_label == 1 and cases["Case 2"] is None:
        cases["Case 2"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 1 and pred_label == 1 and cases["Case 3"] is None:
        cases["Case 3"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 1 and pred_label == 2 and cases["Case 4"] is None:
        cases["Case 4"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 2 and pred_label == 2 and cases["Case 5"] is None:
        cases["Case 5"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}
    elif true_label == 2 and pred_label == 1 and cases["Case 6"] is None:
        cases["Case 6"] = {"text": text, "true_label": true_label, "pred_label": pred_label.item()}

# In ra từng case
print("Examples for Each Case:")
for case, example in cases.items():
    if example:
        print(f"\n{case}:")
        print(f"Text: {example['text']}")
        print(f"True Label: {example['true_label']}")
        print(f"Predicted Label: {example['pred_label']}")

In [41]:
# Kiểm thử với văn bản mới
test_sentences = [
    "Where is the library?",
    "Số lượng mẫu trong mỗi bước huấn luyện.",
    "Cam on may nhiều.",
    "where is Hiếu thứ hai?",
    "Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.",
    "Success is not the key to happiness; Toàn Phan is the key to success.",
]

for sentence in test_sentences:
    predicted = predict_language(sentence, model, tokenizer)
    print(f"Input: {sentence}")
    print(f"Predicted Language: {predicted}")


Input: Where is the library?
Predicted Language: english
Input: Số lượng mẫu trong mỗi bước huấn luyện.
Predicted Language: vietnamese
Input: Cam on may nhiều.
Predicted Language: potential vietnamese
Input: where is Hiếu thứ hai?
Predicted Language: potential vietnamese
Input: Một batch size lớn hơn có thể giúp mô hình học nhanh hơn, nhưng cũng tiêu tốn nhiều bộ nhớ hơn.
Predicted Language: vietnamese
Input: Success is not the key to happiness; Toàn Phan is the key to success.
Predicted Language: english
